In [1]:
%reset -f
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import scipy.io as sio
from scipy.stats import pearsonr

In [2]:
plt.rcParams['font.family'] = ['Times New Roman','SimSun']

In [ ]:
plt.rcParams['font.sans-serif'][0]='Arial'
print(plt.rcParams['font.sans-serif'])

<font size=5>**Prediction accuracy for different cognitive levels and ages**</font>

In [ ]:
cogTasks = ['Fluid intelligence','Emotion expression recognition','PicturePriming','Face recognition','FamousFaces','Motor learning']
predictionPath = r'/path/to/your/data\CamCan\KRR parcel size to cognition\prediction'
BestReps = [46,44,95,97,81,44]
cogTaskNames = ['Gf','EER','PPR','FR','FF','ML']

In [ ]:
fig, axs = plt.subplots(2,3,figsize=(6.9,4),dpi=300)

for ct, ax, rep, ctn in zip(cogTasks, axs.flat, BestReps, cogTaskNames):
    AgeDf = pd.read_csv(rf'/path/to/your/data\CamCan\KRR parcel size to cognition\data input\{ct}.csv')
    Age = np.array(AgeDf['Age']).flatten()
    y = sio.loadmat(f'{predictionPath}\{ct}\{rep}\yOrgAndPredConcat.mat')
    TrueValue = y['y_org_res'].flatten()
    PredValue = y['y_predict_concat'].flatten()
    TrueValueNorm = (TrueValue-np.mean(TrueValue))/np.std(TrueValue)
    TrueValueNorm = TrueValueNorm.flatten()
    PredValueNorm = (PredValue-np.mean(PredValue))/np.std(PredValue)
    PredValueNorm = PredValueNorm.flatten()
    r,p = pearsonr(TrueValue,PredValue)
    if p>0.05:
        lc = 'grey'
    else:
        if r>0:
            lc = 'red'
        else:
            lc = 'blue'

    sns.regplot(x=TrueValueNorm,y=PredValueNorm,scatter=False,line_kws={"color":lc,"lw": 1, "zorder": 1},ax = ax)
    sns.scatterplot(x=TrueValueNorm,y=PredValueNorm,c = Age, cmap = 'Purples',zorder=2,ax = ax,s=3)

    if p>=0.0001:
        text = f"r = {r:.3f}\np = {p:.3e}"  # Format: 3 decimal places, scientific notation
    else:
        text = f"r = {r:.3f}\np < 0.0001"
    
    ax.text(
        x=0.05, y=0.92,  # Position parameters (relative coordinates, 0~1)
        s=text,
        transform=ax.transAxes,  # Key parameter: use relative coordinate system
        fontsize=7,
        color='black',
        va='top',  # Vertical alignment
        ha='left',  # Horizontal alignment
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none')  # Background box
    )
    ax.set_xlabel('True value')
    ax.set_ylabel('Predicted value')
    ax.spines['bottom'].set_linewidth(1)
    ax.spines['left'].set_linewidth(1)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.xaxis.label.set_size(7)
    ax.yaxis.label.set_size(7)
    ax.tick_params(axis='both',width=1,length = 3,labelsize=7)
    ax.set_title(f'{ctn}', size=8)
im = ax.collections[-1]
cax = fig.add_axes([0.96, 0.25, 0.01, 0.5])
cbar = fig.colorbar(
    im, 
    cax=cax, 
    shrink=0.5,
    orientation='vertical',
    # Key modification 1: set tick positions and format
    ticks=[np.min(Age), np.max(Age)],  # Generate 5 evenly spaced ticks (including both ends)
)
cbar.ax.tick_params(axis='y',length = 0,labelsize = 7)
cbar.ax.set_title('Age', y = 1.02, fontsize=8)
plt.tight_layout()
plt.subplots_adjust(left=.07, right=.95,bottom=.07,top=.95,wspace=.3,hspace=.4)

plt.savefig('All Cogtask Accuracy Scatter.svg', format='svg')